In [0]:
%run ./04_Function_Book

In [0]:
# Define the data quality configuration for order_items
# Specifies columns for null checks, regex-based format checks, and composite PK validation
dq_config = {
    "table_name": "orderitems",
    # NULL checks: All listed columns must have 0 nulls
    "null_checks": {
        "order_id": 0,
        "order_item_id": 0,
        "product_id": 0,
        "seller_id": 0,
        "price": 0,
        "shipping_charges": 0
    },
    "format_checks": {
        # REGEX checks: Column patterns, 0 tolerance for violations
        "regex": {
            "order_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "order_item_id": {
                "pattern": r"^[0-9]{1,}$",
                "threshold": 0
            },
            "product_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            },
            "seller_id": {
                "pattern": r"^[A-Za-z0-9]{12,}$",
                "threshold": 0
            }
        },
        # DATATYPE and TIMESTAMP checks not used here
        "datatype_check": {},
        "timestamp_check": {}
    },
    "range_checks": {},
    "date_range_checks": {},
    "enum_checks": {},
    # Composite PK uniqueness: each (order_id, order_item_id) pair must be unique
    "primary_key": {
        "column": ["order_id", "order_item_id"],
        "threshold": 0
    }
}

In [0]:
# Function to aggregate all data quality results into a single metrics DataFrame
# Applies null, format, and PK checks as specified in dq_config
def generate_metrics_df(df, config):
    from pyspark.sql import Row
    from pyspark.sql.functions import current_timestamp
    all_results = []
    all_results += run_null_checks(df, config)
    all_results += run_format_checks(df, config)
    all_results += run_pk_check(df, config)
    # Tag each metrics row with the table name
    for result in all_results:
        result["table_name"] = config["table_name"]
    metrics_df = spark.createDataFrame(Row(**r) for r in all_results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df

In [0]:
# Load the cleaned order items table and run all configured data quality checks
df_order_items = spark.table("retail_catalog.silver.orderitems")
metrics_df = generate_metrics_df(df_order_items, dq_config)

In [0]:
# Return metrics DataFrame to orchestrating/master notebook for consolidation and reporting
metrics_df

DataFrame[column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, table_name: string, run_time: timestamp]